# Scientific Data Analysis: From Pandas to Distributed Computing

This notebook works with real scientific data stored in Parquet format on a shared filesystem. We start with Pandas for single-segment exploration, then scale up with Dask.

**Dataset structure:**
- **lv1** (raw): high-frequency voltage waveforms with nanosecond timestamps
- **lv2** (processed): extracted pulse features — peak amplitude, rise time, pulse length

In [ ]:
from pathlib import Path
import dask.array as da
import dask.dataframe as dd
import psutil
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

print(f"Available RAM: {psutil.virtual_memory().available / (1024 ** 3):.1f} GB")

In [ ]:
lv1_path = Path("/net/pr2/projects/tutorial/2025-09-25-hpda/dataset/lv1/")
lv2_path = Path("/net/pr2/projects/tutorial/2025-09-25-hpda/dataset/lv2/")
print(f"lv1 exists: {lv1_path.exists()}, lv2 exists: {lv2_path.exists()}")

## Part 1: Pandas — Single Segment (lv1)

Parquet supports **predicate pushdown**: filters are applied during file reading, so only matching data is loaded into memory.

In [ ]:
%%time
single_segment_df = pd.read_parquet(lv1_path, filters=[
    ("dataset", "==", "1nA"),
    ("channel_no", "==", 0),
    ("trc_file_no", "==", 0),
    ("segment_no", "==", 0),
])
single_segment_df.head()

In [ ]:
print(f"Memory usage: {single_segment_df.memory_usage(deep=True).sum() / (1024 ** 3):.3f} GB")

### Waveform Visualization

Decimation by factor 40 keeps the plot manageable while preserving the signal shape.

In [ ]:
single_segment_df.groupby(single_segment_df.index // 40).agg(
    {"time_ns": "mean", "voltage_mV": "mean"}
).plot(x="time_ns", y="voltage_mV", kind="scatter", figsize=(10, 5), s=1)
plt.title("lv1: Voltage waveform (decimated 40x)");

In [ ]:
single_segment_df.voltage_mV.hist(bins=100)
plt.xlabel("voltage [mV]")
plt.title("lv1: Voltage distribution");

## Part 2: Pandas — Processed Features (lv2)

Level 2 data contains extracted pulse characteristics: amplitude, rise time, pulse length, polarity.

In [ ]:
df_lv2 = pd.read_parquet(lv2_path, filters=[
    ("dataset", "==", "1nA"),
    ("channel_no", "==", 0),
    ("trc_file_no", "==", 0),
    ("segment_no", "==", 0),
])
df_lv2.head()

In [ ]:
df_lv2.query("peak_length_ns > 1").hist("peak_rise_time_ns", bins=200, range=(0, 2))
plt.title("lv2: Pulse rise time distribution");

## Part 3: Dask — Scaling to Multiple Files

Start a local Dask cluster, then load the same data across multiple trace files in parallel.

In [ ]:
from dask.distributed import Client

# Use your assigned port from 00_setup
DASHBOARD_PORT = 8788  # adjust to your participant number
client = Client(dashboard_address=f":{DASHBOARD_PORT}")
client

### Single Segment via Dask

Same predicate pushdown, but execution is distributed. Compare the result with the Pandas version above.

In [ ]:
%%time
ddf = dd.read_parquet(lv1_path, filters=[
    ("dataset", "==", "64nA"),
    ("channel_no", "==", 0),
    ("trc_file_no", "==", 0),
    ("segment_no", "==", 0),
])
pdf = ddf[["time_ns", "voltage_mV", "trigger_time_ns"]].compute()

In [ ]:
print(f"Memory: {pdf.memory_usage(deep=True).sum() / (1024 ** 3):.3f} GB")
pdf.groupby(pdf.index // 40).agg({"time_ns": "mean", "voltage_mV": "mean"}).plot(
    x="time_ns", y="voltage_mV", kind="scatter", figsize=(10, 5), s=1
)
plt.title("Dask: 64nA waveform");

In [ ]:
pdf.hist("voltage_mV", bins=200)
plt.title("Dask: 64nA voltage distribution");

### Multi-File Processing

Now load multiple trace files at once — this is where Dask shines.

In [ ]:
%%time
ddf = dd.read_parquet(lv1_path, filters=[
    ("dataset", "==", "64nA"),
    ("channel_no", "==", 0),
    ("trc_file_no", "<", 10),
])
print(f"Partitions: {ddf.npartitions}")

In [ ]:
ddf.voltage_mV.mean().compute()

In [ ]:
ddf.voltage_mV.count().compute()

### Distributed Histogram

Two-stage approach: (1) compute range across all partitions, (2) build histogram with consistent binning.

In [ ]:
xmin = ddf.voltage_mV.min().compute()
xmax = ddf.voltage_mV.max().compute()

hist_dask, edges_dask = da.histogram(ddf.voltage_mV, bins=200, range=(xmin, xmax))
hist_comp = hist_dask.compute()

In [ ]:
plt.plot(edges_dask[:-1], hist_comp, drawstyle="steps-post")
plt.yscale("log")
plt.xlabel("voltage [mV]")
plt.ylabel("counts")
plt.title("Distributed histogram: 64nA, 10 trace files")
plt.grid()

## Part 4: Full lv2 Dataset Query

Load the entire lv2 dataset lazily, then apply complex multi-condition filters.

In [ ]:
ddf_lv2 = dd.read_parquet(lv2_path)
ddf_lv2.head()

In [ ]:
%%time
hist_amp, edges_amp = da.histogram(
    ddf_lv2.query('dataset=="0p5nA" and channel_no==0 and peak_length_ns > 0.4 and sign=="negative"').peak_amplitude_mV,
    bins=200, range=(0, 200)
)
hist_amp_comp = hist_amp.compute()

In [ ]:
plt.plot(edges_amp[:-1], hist_amp_comp, drawstyle="steps-post")
plt.yscale("log")
plt.xlabel("peak amplitude [mV]")
plt.ylabel("counts")
plt.title("lv2: peak amplitude (0p5nA, ch0, negative)")
plt.grid()

## Exercise

Analyze dataset `"0p5nA"`, **channel 1** — produce a `peak_amplitude_mV` histogram for negative-sign peaks with `peak_length_ns > 0.4`. Compare the shape with the channel 0 result above.

In [ ]:
# Exercise: histogram for channel 1

# YOUR CODE HERE: use ddf_lv2.query(...) with channel_no==1
# Then da.histogram(...) and plot

In [ ]:
client.close()